# Model search — Aesteel

Ten notebook jest obecnie **jedynym miejscem eksperymentów**. Nie tworzymy jeszcze skryptu produkcyjnego. Testujemy kilka modeli, hiperparametry, liczbę cech i prosty feature importance plot. `test.csv` jest ładowany dopiero w ostatniej komórce i służy wyłącznie do inference.

Zasady: `train.csv` nie ma labeli, więc `val.csv` dzielimy na `reference` i niewidziany `hold-out`. Reference służy wyłącznie do pseudo-labelowania train; hold-out służy do oceny. Model label i model severity są osobne. `ok`/`unknown` zawsze mają `severity=nie_dotyczy`. Isolation Forest pozostaje niezależnym sygnałem anomaly.

**RAW SCORE = 0.75 × Macro-F1(label) + 0.25 × severity accuracy.**


In [ ]:
from pathlib import Path
import sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
TOP_FEATURES_TO_PLOT = 20
FEATURE_COUNTS = [16, 32, 48]
TRIALS_PER_MODEL = 12
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from src.common import LABELS, FAULT_LABELS, clean_spectrum
from src.features import FeatureExtractor
DATA = ROOT / 'data'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)
print('ROOT:', ROOT)


In [ ]:
train = clean_spectrum(pd.read_csv(DATA / 'train.csv'))
val = clean_spectrum(pd.read_csv(DATA / 'val.csv'))
print('train:', train.shape)
print('val:', val.shape)
print(val['label'].value_counts())

val_reference, val_eval = train_test_split(
    val, test_size=0.50, random_state=RANDOM_STATE, stratify=val['label']
)
print('reference:', val_reference.shape)
print('hold-out:', val_eval.shape)


## Feature extraction i ranking cech

Extractor jest fitowany wyłącznie na `train.csv`. Najpierw uczymy pomocniczy XGBoost na wszystkich cechach i tylko wtedy wybieramy TOP-N po `feature_importances_`. Nie używamy hold-out do feature selection.

In [ ]:
def to_matrix(df, names):
    return np.nan_to_num(df[names].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

extractor = FeatureExtractor()
train_f = extractor.fit_transform(train)
ref_f = extractor.transform(val_reference)
eval_f = extractor.transform(val_eval)
all_features = extractor.get_feature_names()

scaler = StandardScaler()
X_train_all = scaler.fit_transform(to_matrix(train_f, all_features)).astype(np.float32)
X_ref_all = scaler.transform(to_matrix(ref_f, all_features)).astype(np.float32)
X_eval_all = scaler.transform(to_matrix(eval_f, all_features)).astype(np.float32)
ref_y = val_reference['label'].astype(str).to_numpy()

classes = [c for c in LABELS if c in set(ref_y)]
proto = np.vstack([np.median(X_ref_all[ref_y == c], axis=0) for c in classes])
dist = np.stack([np.linalg.norm(X_train_all - p, axis=1) for p in proto], axis=1)
pseudo_label = np.asarray(classes)[dist.argmin(axis=1)]

importance_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.035, subsample=0.85,
    colsample_bytree=0.8, min_child_weight=5, reg_alpha=0.2, reg_lambda=3.0,
    gamma=0.05, max_bin=128, tree_method='hist', objective='multi:softprob',
    eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1
)
importance_model.fit(X_train_all, LabelEncoder().fit_transform(pseudo_label))
importance = pd.DataFrame({'feature': all_features, 'importance': importance_model.feature_importances_}).sort_values('importance', ascending=False)
display(importance.head(30))


In [ ]:
plot_df = importance.head(TOP_FEATURES_TO_PLOT).sort_values('importance')
plt.figure(figsize=(10, 7))
plt.barh(plot_df['feature'], plot_df['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top feature importance')
plt.tight_layout()
plt.show()
importance.to_csv(REPORTS / 'feature_importance_all.csv', index=False)


## Pseudo-labels i severity targets

Severity jest drugim, niezależnym targetem. Reference dostarcza prototypy severity, ale sam model severity jest fitowany wyłącznie na pseudo-targetach dla `train.csv`.

In [ ]:
def build_targets(X_train, X_ref, ref_df):
    ref_y = ref_df['label'].astype(str).to_numpy()
    classes = [c for c in LABELS if c in set(ref_y)]
    prototypes = np.vstack([np.median(X_ref[ref_y == c], axis=0) for c in classes])
    d = np.stack([np.linalg.norm(X_train - p, axis=1) for p in prototypes], axis=1)
    pseudo = np.asarray(classes)[d.argmin(axis=1)]
    confidence = np.exp(-d + d.min(axis=1, keepdims=True))
    confidence = confidence.max(axis=1)

    sev_names = ['male', 'srednie', 'duze']
    severity_y = np.full(len(X_train), 'nie_dotyczy', dtype=object)
    for fault in FAULT_LABELS:
        tm = pseudo == fault
        rm = ref_y == fault
        if not tm.any() or not rm.any(): continue
        protos, names = [], []
        for sev in sev_names:
            m = rm & (ref_df['severity'].astype(str).to_numpy() == sev)
            if m.any(): protos.append(np.median(X_ref[m], axis=0)); names.append(sev)
        if not protos: continue
        sd = np.stack([np.linalg.norm(X_train[tm] - p, axis=1) for p in protos], axis=1)
        severity_y[tm] = np.asarray(names)[sd.argmin(axis=1)]
    return pseudo, severity_y, confidence

pseudo_label, pseudo_severity, pseudo_conf = build_targets(X_train_all, X_ref_all, val_reference)
print('pseudo labels:', pd.Series(pseudo_label).value_counts().to_dict())
print('pseudo severity:', pd.Series(pseudo_severity).value_counts().to_dict())
print('mean pseudo confidence:', round(float(pseudo_conf.mean()), 4))


## Parameter search

Porównujemy kilka rodzin modeli. Dla każdego trialu trenujemy osobny model label oraz osobny model severity. W finalnym score severity jest liczona jako `nie_dotyczy` dla `ok/unknown`. Test nie bierze udziału w searchu.

Na start jest 12 losowych konfiguracji na model. Po pierwszym uruchomieniu możesz zwiększyć `TRIALS_PER_MODEL`.

In [ ]:
MODEL_SPACES = {
 'xgboost': {
   'n_estimators':[180,260,360,480], 'max_depth':[3,4,5], 'learning_rate':[0.025,0.035,0.05],
   'subsample':[0.75,0.85,0.95], 'colsample_bytree':[0.65,0.8,0.95], 'min_child_weight':[3,5,8],
   'reg_alpha':[0.0,0.2,0.5], 'reg_lambda':[2.0,3.0,5.0], 'gamma':[0.0,0.05,0.15], 'max_bin':[64,128,256]
 },
 'extra_trees': {
   'n_estimators':[200,400,700], 'max_depth':[None,8,12,18], 'min_samples_leaf':[1,2,4,8],
   'max_features':[0.5,0.75,1.0], 'class_weight':['balanced']
 },
 'random_forest': {
   'n_estimators':[200,400,700], 'max_depth':[None,8,12,18], 'min_samples_leaf':[1,2,4,8],
   'max_features':['sqrt',0.6,0.9], 'class_weight':['balanced']
 },
 'hist_gradient_boosting': {
   'max_iter':[150,250,400], 'learning_rate':[0.03,0.05,0.08], 'max_leaf_nodes':[15,31,63],
   'max_depth':[None,4,6], 'min_samples_leaf':[10,20,40], 'l2_regularization':[0.5,1.0,3.0]
 }
}

def make_model(name, params, n_classes):
    if name == 'xgboost':
        return xgb.XGBClassifier(**params, objective='multi:softprob', num_class=n_classes, eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1)
    if name == 'extra_trees': return ExtraTreesClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    if name == 'random_forest': return RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    return HistGradientBoostingClassifier(**params, random_state=RANDOM_STATE)

def run_trial(name, params, feature_count):
    selected = importance.head(feature_count)['feature'].tolist()
    idx = [all_features.index(f) for f in selected]
    Xt, Xe = X_train_all[:, idx], X_eval_all[:, idx]
    label_enc = LabelEncoder().fit(pseudo_label)
    y = label_enc.transform(pseudo_label)
    label_model = make_model(name, params, len(label_enc.classes_))
    label_model.fit(Xt, y)
    pred_label = label_enc.inverse_transform(label_model.predict(Xe).astype(int))

    severity_mask = pseudo_severity != 'nie_dotyczy'
    sev_enc = LabelEncoder().fit(['male','srednie','duze'])
    sev_model = make_model(name, params, 3)
    sev_model.fit(Xt[severity_mask], sev_enc.transform(pseudo_severity[severity_mask]))
    pred_sev = np.where(np.isin(pred_label, FAULT_LABELS), sev_enc.inverse_transform(sev_model.predict(Xe).astype(int)), 'nie_dotyczy')

    true_label = val_eval['label'].astype(str).to_numpy()
    true_sev = np.where(np.isin(true_label, FAULT_LABELS), val_eval['severity'].astype(str).to_numpy(), 'nie_dotyczy')
    acc = accuracy_score(true_label, pred_label)
    macro = f1_score(true_label, pred_label, labels=LABELS, average='macro', zero_division=0)
    sev_acc = accuracy_score(true_sev, pred_sev)
    score = 0.75 * macro + 0.25 * sev_acc
    return {'model':name, 'features':feature_count, 'accuracy':acc, 'macro_f1':macro, 'severity_accuracy':sev_acc, 'raw_score':score, **params}, label_model, sev_model


In [ ]:
results = []
best_models = {}
for model_name, space in MODEL_SPACES.items():
    print(f'\n=== {model_name} ===')
    for trial, params in enumerate(ParameterSampler(space, n_iter=TRIALS_PER_MODEL, random_state=RANDOM_STATE), 1):
        for feature_count in FEATURE_COUNTS:
            t0 = time.time()
            row, label_model, sev_model = run_trial(model_name, params, feature_count)
            row['trial'] = trial; row['seconds'] = time.time() - t0
            results.append(row)
        best_now = max([r for r in results if r['model']==model_name], key=lambda r:r['raw_score'])
        print(f"trial={trial:02d} best={best_now['raw_score']:.4f} (features={best_now['features']})")

results_df = pd.DataFrame(results).sort_values(['raw_score','macro_f1'], ascending=False).reset_index(drop=True)
display(results_df.head(25))
results_df.to_csv(REPORTS / 'parameter_search_notebook.csv', index=False)


## Porównanie modeli

Tutaj patrzymy na **raw score**, a nie tylko accuracy. To ważne, bo konkurs premiuje Macro-F1 i nie chcemy modelu, który osiąga wysoką accuracy tylko przez dominującą klasę.

In [ ]:
summary = (results_df.groupby('model').agg(best_raw_score=('raw_score','max'), best_macro_f1=('macro_f1','max'), best_accuracy=('accuracy','max'), best_severity_accuracy=('severity_accuracy','max')).sort_values('best_raw_score', ascending=False))
display(summary)
best = results_df.iloc[0]
print('OVERALL BEST')
print(best.to_string())


## Feature importance najlepszego modelu

Po searchu ponownie fitujemy najlepszy model label na jego wybranym TOP-N i rysujemy prosty poziomy wykres ważności cech.

In [ ]:
best_name = best['model']
best_features_n = int(best['features'])
best_params = {k: best[k] for k in MODEL_SPACES[best_name].keys() if k in best and pd.notna(best[k])}
selected = importance.head(best_features_n)['feature'].tolist()
idx = [all_features.index(f) for f in selected]
best_enc = LabelEncoder().fit(pseudo_label)
best_model = make_model(best_name, best_params, len(best_enc.classes_))
best_model.fit(X_train_all[:, idx], best_enc.transform(pseudo_label))
best_imp = pd.DataFrame({'feature':selected, 'importance':best_model.feature_importances_}).sort_values('importance', ascending=False)
plot = best_imp.head(TOP_FEATURES_TO_PLOT).sort_values('importance')
plt.figure(figsize=(10,7))
plt.barh(plot['feature'], plot['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title(f'Feature importance — {best_name}, TOP {best_features_n}')
plt.tight_layout(); plt.show()
best_imp.to_csv(REPORTS / 'best_feature_importance.csv', index=False)


## Test — dopiero po zakończeniu searchu

Ta sekcja nie bierze udziału w żadnym wyborze modelu. Dopiero po zamrożeniu konfiguracji wczytujemy `test.csv` i wykonujemy inference.

In [ ]:
test = clean_spectrum(pd.read_csv(DATA / 'test.csv'))
test_f = extractor.transform(test)
X_test_all = scaler.transform(to_matrix(test_f, all_features)).astype(np.float32)
X_test = X_test_all[:, idx]
test_label = best_enc.inverse_transform(best_model.predict(X_test).astype(int))

# Severity model is separate from label model.
sev_enc = LabelEncoder().fit(['male','srednie','duze'])
sev_model = make_model(best_name, best_params, 3)
sev_mask = pseudo_severity != 'nie_dotyczy'
sev_model.fit(X_train_all[sev_mask][:, idx], sev_enc.transform(pseudo_severity[sev_mask]))
test_severity = np.where(np.isin(test_label, FAULT_LABELS), sev_enc.inverse_transform(sev_model.predict(X_test).astype(int)), 'nie_dotyczy')

iso = IsolationForest(n_estimators=240, contamination='auto', random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X_train_all[:, idx])
test_anomaly = np.clip(-iso.decision_function(X_test), -1, 1)

test_predictions = test[['engine_id','cylinder']].copy()
test_predictions['label'] = test_label
test_predictions['severity'] = test_severity
test_predictions['anomaly_score'] = test_anomaly
test_predictions.to_csv(ROOT / 'outputs' / 'test_predictions_search.csv', index=False)
display(test_predictions.head(20))
print('Saved:', ROOT / 'outputs' / 'test_predictions_search.csv')
print('Sanity:', len(test_predictions), 'predictions for', len(test), 'test rows')
